In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [10]:
# 1. Chargement des données propres
df = pd.read_csv('../dataset/traite/train_cleaned.csv')

In [11]:
# 2. Séparation Features (X) / Target (y)
y = df['SalePrice']
X = df.drop('SalePrice', axis=1)

In [12]:
# 3. Définition de la méthode de validation (Cross-Validation)
# On découpe en 5 parts (folds), on mélange (shuffle) pour éviter les biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Fonction pour calculer le score RMSE (Root Mean Squared Error)
# Plus le chiffre est bas, meilleur est le modèle.
def cv_rmse(model):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kf))
    return rmse

In [ ]:

# rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kf))

# Voici ce qui se passe, de l'intérieur vers l'extérieur :

# 1. Le Cœur : cross_val_score(...)
# C'est la fonction de Scikit-Learn qui automatise la "torture" du modèle.
# Elle ne va pas entraîner le modèle une fois, mais 5 fois (car cv=kf et on a défini kf avec 5 splits).

# Tour 1 : Elle entraîne sur les parties 1,2,3,4 et teste sur la 5.

# Tour 2 : Elle entraîne sur 1,2,3,5 et teste sur la 4.

# etc.
# Elle renvoie donc une liste de 5 notes.

# 2. Le Paramètre Bizarre : scoring="neg_mean_squared_error"
# C'est ici que tout le monde bloque la première fois. Pourquoi "neg" (négatif) ?

# La logique de Scikit-Learn : Dans cette librairie, par convention, "Un score plus élevé est TOUJOURS mieux".

# Pour la précision (Accuracy), 95% est mieux que 80%. C'est logique.

# Le problème de l'Erreur : Pour nous, l'erreur (MSE), plus c'est bas, mieux c'est. (Une erreur de 0 est parfaite, une erreur de 1000 est nulle).

# Le Conflit : Scikit-Learn veut maximiser, nous voulons minimiser.

# La Solution de Scikit-Learn : Pour respecter sa propre règle ("plus haut = mieux"), Scikit-Learn calcule l'erreur (ex: 0.15) et la multiplie par -1.

# L'erreur devient -0.15.

# Si le modèle s'améliore, l'erreur passe à 0.10, ce qui devient -0.10.

# Mathématiquement, -0.10 est "plus grand" (plus haut sur l'axe) que -0.15.

# Donc Scikit-Learn est content : le score a "augmenté".

# Résultat intermédiaire : cross_val_score nous renvoie une liste de nombres négatifs : [-0.12, -0.11, -0.13, ...].

# 3. Le Signe Moins : -cross_val_score
# Puisque Scikit-Learn nous a donné des nombres négatifs par convention, nous devons les remettre en positif pour que cela ait du sens pour nous, les humains.

# Mathématiquement : - (-0.12) = +0.12.

# On retrouve notre Mean Squared Error (MSE) positive.

# 4. La Racine Carrée : np.sqrt(...)
# MSE = Mean Squared Error (Erreur Moyenne au Carré).

# Si ton erreur est de 10$, la MSE est de 100 ($²$). C'est difficile à interpréter.

# On prend la racine carrée (Square Root) pour revenir à l'unité d'origine.

# $\sqrt{MSE} = RMSE$ (Root Mean Squared Error).

# En résumé
# Cette fonction fait ceci :

# Elle lance 5 entraînements.

# Pour chaque entraînement, elle calcule l'erreur au carré.

# Elle met un signe "moins" pour satisfaire Scikit-Learn.

# Nous remettons le signe positif.

# Nous prenons la racine carrée pour avoir une "vraie" erreur moyenne.

# Elle retourne une liste de 5 erreurs RMSE.

# C'est pour cela que quand on l'appelle, on fait .mean() (pour avoir la moyenne des 5 tours) et .std() (pour voir si le modèle est stable ou s'il change d'avis tout le temps).

# Est-ce que c'est parfaitement clair maintenant ? C'est crucial pour interpréter ton score de 0.1124 (qui est donc la moyenne de ces racines carrées).

In [ ]:
# --- Modèle 1 : Régression Linéaire (Ridge) ---
# alpha est la force de la régularisation.
# alpha = 10 est souvent un bon point de départ pour ce dataset.
ridge_model = Ridge(alpha=10)

score_ridge = cv_rmse(ridge_model)
print(f"Ridge (Linear) Score: {score_ridge.mean():.4f} (std: {score_ridge.std():.4f})")


Ridge (Linear) Score: 0.1152 (std: 0.0171)


In [18]:
# --- Modèle 2 : Random Forest ---
# n_estimators : nombre d'arbres (100 est standard, on peut monter à 500 pour plus de précision)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 utilise tous tes coeurs CPU

score_rf = cv_rmse(rf_model)
print(f"Random Forest Score: {score_rf.mean():.4f} (std: {score_rf.std():.4f})")


Random Forest Score: 0.1327 (std: 0.0149)


In [ ]:


# Régression Linéaire Multiple Standard (Sans filet de sécurité)
lr_model = LinearRegression()

score_lr = cv_rmse(lr_model)
print(f"Linear Regression (OLS) Score: {score_lr.mean():.4f} (std: {score_lr.std():.4f})")


Linear Regression (OLS) Score: 0.1243 (std: 0.0167)


In [16]:
import xgboost as xgb

# --- Modèle 3 : XGBoost ---
# XGBoost a beaucoup d'hyperparamètres. Voici une configuration robuste de base.
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', # Notre but est de minimiser l'erreur carrée
    n_estimators=1000,            # Beaucoup d'arbres
    learning_rate=0.05,           # On apprend doucement (plus précis)
    max_depth=3,                  # Arbres peu profonds pour éviter le sur-apprentissage
    colsample_bytree=0.7,         # On n'utilise que 70% des colonnes par arbre (diversité)
    random_state=42,
    n_jobs=-1
)

print("Entraînement XGBoost en cours... (patience)")
score_xgb = cv_rmse(xgb_model)
print(f"XGBoost Score: {score_xgb.mean():.4f} (std: {score_xgb.std():.4f})")


Entraînement XGBoost en cours... (patience)
XGBoost Score: 0.1124 (std: 0.0153)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# On génère les prédictions finales sur tout le dataset (pour l'exemple visuel)
# Idéalement on fait ça sur un Test set séparé, mais ici on veut voir le comportement global
y_pred_final = blended_predictions(X)

plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred_final, alpha=0.5, color='blue')

# On trace la diagonale parfaite
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=3)

plt.xlabel('Vrai Prix (Log)')
plt.ylabel('Prix Prédit (Log)')
plt.title('Vérité vs Prédiction : Sommes-nous sur la ligne ?')
plt.show()
